# Bank Marketing — Exploratory Data Analysis

UCI Bank Marketing 数据集的探索性分析，了解数据分布、特征关系及目标变量的预测难度。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

## 1. 目标变量分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = train['subscribe'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Target Distribution (Count)')
axes[0].set_xlabel('Subscribe')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

pcts = train['subscribe'].value_counts(normalize=True) * 100
wedges, texts, autotexts = axes[1].pie(
    pcts.values, labels=['No', 'Yes'], autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'], explode=(0, 0.05)
)
for t in autotexts:
    t.set_fontweight('bold')
axes[1].set_title('Target Distribution (%)')

plt.tight_layout()
plt.show()

print(f'Imbalance ratio (no/yes): {pcts["no"] / pcts["yes"]:.1f}:1')

## 2. 数值特征分布

In [ ]:
num_cols = ['age', 'duration', 'campaign', 'pdays', 'previous',
            'emp_var_rate', 'cons_price_index', 'cons_conf_index',
            'lending_rate3m', 'nr_employed']

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    train[col].hist(bins=40, ax=ax, color='#3498db', edgecolor='white', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel('')

axes[-1].axis('off')
axes[-2].axis('off')
plt.tight_layout()
plt.show()

## 3. 类别不平衡下的特征区分度

In [ ]:
train_encoded = train.copy()
train_encoded['subscribe_binary'] = train_encoded['subscribe'].map({'yes': 1, 'no': 0})

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
key_features = ['age', 'duration', 'campaign', 'previous', 'pdays', 'emp_var_rate']

for i, col in enumerate(key_features):
    ax = axes[i // 3][i % 3]
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = train_encoded[train_encoded['subscribe_binary'] == label][col]
        subset = subset[subset < subset.quantile(0.99)]
        ax.hist(subset, bins=30, alpha=0.5, label='Yes' if label else 'No', color=color)
    ax.set_title(f'{col} by Subscribe')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. 分类特征与目标关系

In [ ]:
cat_cols = ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'poutcome']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    ct = pd.crosstab(train[col], train['subscribe'], normalize='index') * 100
    ct['no'] = 100 - ct['yes']
    ct = ct.sort_values('yes', ascending=True)
    bars = ct.plot(kind='barh', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'], width=0.8)
    ax.set_title(f'{col} → Subscribe Rate')
    ax.set_xlabel('Percentage (%)')
    ax.legend(loc='lower right')
    for container in ax.containers:
        labels = [f'{w:.0f}%' if (w := v.get_width()) > 5 else '' for v in container]
        ax.bar_label(container, labels=labels, label_type='center', fontsize=8)

axes[-1].axis('off')
axes[-2].axis('off')
plt.tight_layout()
plt.show()

## 5. 数值特征相关性热力图

In [ ]:
num_cols_avail = [c for c in num_cols if c in train.columns]
corr = train[num_cols_avail].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Numeric Feature Correlations')
plt.tight_layout()
plt.show()

## 6. 关键发现总结

| 发现 | 影响 |
|------|------|
| 严重类别不平衡（87:13） | Accuracy 不可靠，应关注 Precision/Recall/F1 |
| `duration` 与目标强相关 | 有数据泄漏风险（通话结束才知道时长），基础模型必须移除 |
| `pdays=999` 占比大 | 需特殊处理：999 → NaN 或新增 `contacted_before` 特征 |
| 分类特征大量 `unknown` | 必须处理缺失值，不能忽略 |
| 经济指标高度相关 | `emp_var_rate` / `nr_employed` / `cons_price_index` 相关性 > 0.8 |
| `poutcome=success` 客户转化率高 | 历史成功是强信号，应在特征工程中利用 |
| `month` / `day_of_week` 存在周期性 | 应使用 sin/cos 编码而非 OneHot |